# A Complete RAG Pipeline with LangChain

Wiring up actual retrieval-augmented generation for the Insurellm assistant,
using the Chroma vector store built in the previous notebook (chunked
documents, embedded with Hugging Face's `all-MiniLM-L6-v2`).

This notebook uses LangChain to connect the pieces: a **retriever** (pulls
relevant chunks from the vector store for a given question) and an **LLM**
(answers using those chunks as context). The result is a working
question-answering assistant, wrapped in a Gradio chat interface.


## Setup


In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr


In [ ]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)


## Connecting to the existing vector store

Reconnecting to the Chroma database built previously, using the same
embedding model (`all-MiniLM-L6-v2`) it was originally created with -- the
embedding model has to match, since vectors from a different model wouldn't be
comparable.


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)


## The two key LangChain objects: retriever and LLM

`vectorstore.as_retriever()` wraps the vector store in LangChain's standard
retriever interface -- given a query, it returns the most relevant chunks.

`ChatOpenAI` wraps the chat model. `temperature=0` is chosen deliberately here:
for a factual assistant answering questions about company data, I want the
most likely answer every time, not variety.

### A note on temperature

Temperature controls how the model selects the next token during generation,
not "creativity" in any literal sense:

- `temperature=0` -- always select the highest-probability token. Output is
  as close to deterministic as this setting can make it.
- `temperature=1` -- sample proportionally to probability (a token with 10%
  probability gets picked roughly 10% of the time), producing more varied
  output.

Two caveats worth keeping in mind:
- `temperature=0` alone doesn't guarantee fully reproducible output --
  reproducibility also requires fixing a random seed, and even then isn't
  always guaranteed.
- If varied, creative output is actually the goal, that's better controlled
  through the system prompt than through temperature.


In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)


## Trying each piece independently

Both `retriever` and `llm` implement LangChain's standard `invoke()` method.
Worth calling each on its own first, to see clearly what each one actually
returns before combining them.


In [ ]:
retriever.invoke("Who is Avery?")


In [ ]:
llm.invoke("Who is Avery?")


The retriever returns actual document chunks from the knowledge base. The bare
LLM call, with no context supplied, can only guess or admit it doesn't know --
this is exactly the gap RAG is meant to close.


## Combining retrieval and generation

The system prompt has a `{context}` placeholder, filled in per-question with
whatever the retriever finds. This is the actual RAG mechanism: the model
never sees the whole knowledge base, only the specific chunks relevant to the
current question.


In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""


`answer_question` ties it together: retrieve relevant chunks for the question,
join their text into a single context block, fill the system prompt template
with it, and send both the system and user messages to the LLM.


In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content


In [ ]:
answer_question("Who is Averi Lancaster?", [])


## Trying it in Gradio

`answer_question` already matches the `(message, history)` signature
`gr.ChatInterface` expects, so it plugs in directly with no extra wrapping.


In [ ]:
gr.ChatInterface(answer_question).launch()


## Summary

This is a complete, working RAG pipeline: chunked documents, embedded and
stored in Chroma, retrieved per-question, and fed into an LLM as context. Worth
noting how little code the actual retrieval-and-generation logic takes once
the vector store already exists -- most of the earlier complexity was in
building that store, not in querying it.

Possible next steps:
- Compare retrieval quality against the brute-force keyword approach from the
  first notebook, on questions that don't contain an exact dictionary-key
  match.
- Try a different embedding model, or `OpenAIEmbeddings`, and compare answer
  quality.
- Add source citations to the response, showing which document chunks were
  used to answer a given question.
